# ТЕМА 6. Агенти та інструменти — перевірена навчальна версія

У цій темі ви створите агентів за допомогою **LangChain v1** (під капотом використовується LangGraph), додасте інструменти, керування станом і інтеграцію із зовнішніми сервісами.

> **Що виправлено:** застарілі виклики `create_react_agent` замінено на `create_agent`; додано актуальні `ToolRuntime` і middleware; усунено `eval`; прибрано захардкоджені API-ключі; демонстраційні дані відокремлено від реальних. Комірки з платними запитами вимагають дійсного OpenAI API-ключа; зовнішній пошук — Tavily API-ключа. Не всі приклади запускають LLM автоматично.

**Порядок:** виконуй комірки зверху вниз. Після перезапуску Colab повторно виконай встановлення пакетів, налаштування секретів і глобальні змінні.

### Встановлення пакетів

In [ ]:
# Встановлюємо основні бібліотеки та окремий пакет інтеграції Tavily.
# -q = quiet: pip не друкує звичайний прогрес встановлення.
%pip install -q -U langgraph langchain langchain-openai langchain-tavily

### Налаштування API ключів

In [ ]:
import os
from google.colab import userdata

# Значення беруться з Google Colab → Секрети (значок ключа ліворуч).
# Увімкни "Доступ до записника" для відповідних секретів.
# Не вставляй самі секретні значення у код або вивід комірок.
for secret_name in ("OPENAI_API_KEY", "LANGSMITH_API_KEY", "TAVILY_API_KEY"):
    secret_value = userdata.get(secret_name)
    if secret_value:
        os.environ[secret_name] = secret_value
    else:
        print(f"Увага: секрет {secret_name} не знайдено або він порожній")

# LangSmith tracing — опційний. Сам ключ не вмикає трасування автоматично.
# Щоб записувати траси у LangSmith, розкоментуй:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "AI_topic_6_lesson"

In [ ]:
# Перевірка присутності ключів БЕЗ друку секретних значень.
for key in ("OPENAI_API_KEY", "LANGSMITH_API_KEY", "TAVILY_API_KEY"):
    print(f"{key}: {'OK' if os.getenv(key) else 'MISSING'}")

### Створення глобальних змінних

In [ ]:
# Централізована конфігурація: редагуй назви моделей лише тут.
# У поточному ноутбуці MODEL_NAME — базова модель, ADVANCED_MODEL_NAME — для довгих діалогів.
# Переконайся, що обидві моделі доступні у твоєму OpenAI API-проєкті.
MODEL_NAME = "gpt-6-luna"
ADVANCED_MODEL_NAME = "gpt-6.1-sol"
MESSAGE_THRESHOLD = 10

# Температура підтримується не з усіма режимами reasoning.
# Для прикладу нижче використано reasoning_effort="none".
TEMPERATURE = 0.2
MAX_TOKENS = 1000  # Максимум вихідних токенів моделі (у тому числі, де застосовно, reasoning).
TIMEOUT = 30        # Тайм-аут одного API-запиту в секундах.

### 1. Базове створення агента

Створення базового агента з рядковим ідентифікатором моделі


In [ ]:
from langchain.agents import create_agent

# У LangChain v1 замість застарілого create_react_agent використовуємо create_agent.
# Без tools це приклад звичайного LLM-виклику в агентній обгортці.
basic_agent = create_agent(
    model=f"openai:{MODEL_NAME}",
    tools=[],
)

Виклик агента з тестовим запитом


In [ ]:
# Викликаємо саме basic_agent: інші приклади не перезаписують цю змінну.
basic_response = basic_agent.invoke({
    "messages": [{"role": "user", "content": "Hello!"}]
})
print(basic_response["messages"][-1].content)

### 2. Агент з налаштуваннями моделі

Налаштування моделі з температурою та max_tokens


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Об'єкт ChatOpenAI дає змогу задати параметри генерації явно.
# Для даної конфігурації температура використовується з reasoning_effort="none".
configured_model = ChatOpenAI(
    model=MODEL_NAME,
    reasoning_effort="none",
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    timeout=TIMEOUT,
)
configured_agent = create_agent(model=configured_model, tools=[])
print("Configured model:", configured_model.model_name)

### 3. Динамічний вибір моделі

Реалізація **middleware** для вибору моделі. Для коротких розмов використовується `MODEL_NAME`, для довших за `MESSAGE_THRESHOLD` — `ADVANCED_MODEL_NAME`. Кількість повідомлень — навчальне правило, а не реальна оцінка складності завдання. Без checkpointer у кожному незалежному `invoke()` історія не накопичується.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest
from langchain_openai import ChatOpenAI

# Створюємо об'єкти один раз, а не перед кожним викликом.
routing_basic_model = ChatOpenAI(model=MODEL_NAME)
routing_advanced_model = ChatOpenAI(model=ADVANCED_MODEL_NAME)

@wrap_model_call
def model_selector(request: ModelRequest, handler):
    """Між 2 моделями обираємо за кількістю повідомлень у поточному стані."""
    count = len(request.state["messages"])
    selected_model = (routing_advanced_model if count > MESSAGE_THRESHOLD
                      else routing_basic_model)
    print(f"Messages: {count}; selected model: {selected_model.model_name}")
    # override створює новий запит; змінювати request.model напряму не потрібно.
    return handler(request.override(model=selected_model))

dynamic_agent = create_agent(
    model=routing_basic_model,
    tools=[],
    middleware=[model_selector],
)

# Цей одноразовий запит має 1 повідомлення, тому обирає базову модель.
# Для перевірки другої гілки передай > MESSAGE_THRESHOLD повідомлень
# або використовуй checkpointer і той самий thread_id у наступних викликах.
dynamic_response = dynamic_agent.invoke({
    "messages": [{"role": "user", "content": "Hello!"}]
})
print(dynamic_response["messages"][-1].content)

### 4. Агент з інструментами

Створення агента з двома інструментами. `search_demo` — **заглушка**, не реальний вебпошук. `calculator` обчислює арифметику без небезпечного `eval()`; передавання виразу в LLM не гарантує, що модель завжди вирішить викликати інструмент.

In [ ]:
import ast
import operator
from langchain_core.tools import tool

@tool
def search_demo(query: str) -> str:
    """Демонстрація пошуку: повертає тестовий рядок, НЕ звертається до інтернету."""
    return f"[DEMO] Умовні результати пошуку для: {query}"

# Безпечний навчальний калькулятор: дозволені лише числа і +, -, *, /, **.
# eval(expression) НЕ використовується, бо він може виконати довільний Python-код.
_OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub,
              ast.Mult: operator.mul, ast.Div: operator.truediv,
              ast.Pow: operator.pow, ast.USub: operator.neg,
              ast.UAdd: operator.pos}

def _calculate_node(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        if abs(node.value) > 1_000_000:
            raise ValueError("Завелике число")
        return node.value
    if isinstance(node, ast.UnaryOp) and type(node.op) in (ast.UAdd, ast.USub):
        return _OPERATORS[type(node.op)](_calculate_node(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _OPERATORS:
        left, right = _calculate_node(node.left), _calculate_node(node.right)
        if isinstance(node.op, ast.Pow) and (abs(right) > 10 or abs(left) > 1_000):
            raise ValueError("Завеликий степінь")
        return _OPERATORS[type(node.op)](left, right)
    raise ValueError("Дозволені лише числа і арифметичні операції")

@tool
def calculator(expression: str) -> str:
    """Безпечно обчислити короткий арифметичний вираз, наприклад 44*3.5411."""
    try:
        if len(expression) > 100:
            raise ValueError("Вираз завеликий")
        return str(_calculate_node(ast.parse(expression, mode="eval").body))
    except (ArithmeticError, SyntaxError, ValueError, OverflowError) as exc:
        return f"Помилка обчислення: {exc}"

# Інструменти мають назви, описи та схеми аргументів для tool calling.
tools_agent = create_agent(model=configured_model, tools=[search_demo, calculator])
print("Calculator direct test:", calculator.invoke({"expression": "44*3.5411"}))

Тестування агента з інструментами


In [ ]:
# Агент сам може вирішити викликати calculator; прямий виклик вище — детермінований.
tools_response = tools_agent.invoke({
    "messages": [{"role": "user", "content": "Обчисли 44*3.5411 за допомогою calculator."}]
})
print(tools_response["messages"][-1].content)

### 5. Системний промпт

Налаштування системного промпту для агента


In [ ]:
# System prompt визначає інструкції, а не замінює повідомлення користувача.
system_message = "Ти — корисний асистент, який завжди відповідає українською мовою."
prompt_agent = create_agent(
    model=configured_model,
    tools=[],
    system_prompt=system_message,
)

### 6. Структурований вихід

Опис структурованої відповіді за допомогою Pydantic. `response_format=StructuredResponse` застосовується до **агента**, а `structured_response` містить результат. Поле `confidence` — самооцінка моделі, а не гарантована статистична ймовірність.

In [ ]:
from pydantic import BaseModel, Field

class StructuredResponse(BaseModel):
    answer: str = Field(description="Відповідь на запитання")
    confidence: float = Field(ge=0, le=1, description="Оцінка впевненості моделі від 0 до 1")
    sources: list[str] = Field(description="Посилання на реально використані джерела; [] якщо їх немає")

# Structured output всередині LangChain v1: response_format — схема фінального результату.
structured_agent = create_agent(
    model=configured_model,
    tools=[],
    response_format=StructuredResponse,
    system_prompt=("Відповідай українською. Не вигадуй джерела. "
                   "Оцінка confidence є суб'єктивною, не каліброваною ймовірністю."),
)
structured_result = structured_agent.invoke({
    "messages": [{"role": "user", "content": "Яка столиця України?"}]
})
print(structured_result["structured_response"])

### 7. Кастомний стан агента

Кастомний стан (`AgentState`) із власними полями. Для збереження історії між викликами потрібні `checkpointer` і постійний `thread_id`. `InMemorySaver` зберігає стан тільки в поточній сесії.

In [ ]:
from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Кастомний стан розширює базове поле messages.
# Додаткові значення передаються в agent.invoke(...).
class CustomState(AgentState):
    user_id: str
    session_count: int

# InMemorySaver зберігає стан за thread_id тільки доки працює процес Colab.
state_checkpointer = InMemorySaver()
state_agent = create_agent(
    model=configured_model,
    tools=[],
    state_schema=CustomState,
    checkpointer=state_checkpointer,
)

# Приклад виклику (розкоментуй, щоб витратити токени API):
# state_result = state_agent.invoke(
#     {"messages": [{"role": "user", "content": "Привіт"}],
#      "user_id": "user123", "session_count": 1},
#     config={"configurable": {"thread_id": "lesson-thread-1"}},
# )

### 8. Створення інструментів

Базове визначення інструмента з декоратором


In [ ]:
from langchain_core.tools import tool

@tool
def get_weather_basic(city: str) -> str:
    """Демо прогнозу: повертає вигадані навчальні дані, не поточну погоду."""
    return f"[DEMO, не фактична погода] {city}: сонячно, 22°C"

print("Tool name:", get_weather_basic.name)
print("Example:", get_weather_basic.invoke({"city": "Київ"}))

### 9. Власна назва інструмента

Встановлення власної назви для інструмента


In [ ]:
@tool("weather_forecast")
def get_weather_named(city: str) -> str:
    """Демо прогнозу з власною назвою інструмента; фактичного API немає."""
    return f"[DEMO] Прогноз для {city} (умовний)"

print("Custom tool name:", get_weather_named.name)

### 10. Власний опис інструмента

Додавання детального опису до інструмента


In [ ]:
@tool
def get_weather_detailed(city: str) -> str:
    """Отримати ДЕМОНСТРАЦІЙНИЙ опис погоди для міста.

    Цей інструмент НЕ звертається до погодного API і НЕ повертає живі дані.
    """
    return f"[DEMO] Детальна погода для {city}: дані не завантажувалися"

print("Description:", get_weather_detailed.description)

### 11. Складні схеми вводу з Pydantic

Визначення Pydantic схеми для параметрів інструмента


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class WeatherQuery(BaseModel):
    city: str = Field(min_length=1, description="Назва міста")
    units: Literal["celsius", "fahrenheit"] = "celsius"
    days: int = Field(default=1, ge=1, le=7, description="Кількість днів: 1–7")

@tool(args_schema=WeatherQuery)
def get_weather_with_schema(city: str, units: str = "celsius", days: int = 1) -> str:
    """Демонстраційний прогноз без реального звернення до погодного API."""
    return f"[DEMO] Прогноз для {city} на {days} дн., одиниці: {units}"

# Pydantic валідовує аргументи перед передаванням до функції.
print(get_weather_with_schema.invoke({"city": "Львів", "days": 2}))

### 12. Доступ до стану агента

У LangChain v1 `ToolRuntime` надає інструменту доступ до **стану агента** через `runtime.state`. Він автоматично передається фреймворком і не входить до JSON-аргументів, які генерує LLM.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_context(runtime: ToolRuntime) -> str:
    """Повернути кількість повідомлень із поточного стану агента."""
    # ToolRuntime ін'єктується LangChain автоматично; LLM не передає цей параметр.
    return f"Повідомлень в історії: {len(runtime.state['messages'])}"

# Виклик через агент потрібен для автоматичної ін'єкції runtime.
# context_agent = create_agent(model=configured_model, tools=[get_context])

### 13. Доступ до специфічного поля стану

Інструмент з доступом до окремого поля


In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_id(runtime: ToolRuntime) -> str:
    """Отримати user_id з кастомного стану агента, якщо він заданий."""
    user_id = runtime.state.get("user_id")
    return f"User ID: {user_id}" if user_id else "user_id відсутній у стані"

# Для доступу до user_id агенту потрібно передати state_schema=CustomState.

### 14. Оновлення стану з інструмента

Для оновлення стану інструмент повертає `Command(update=...)`. Якщо оновлюються повідомлення, додається `ToolMessage` із коректним `tool_call_id`, щоб цикл tool calling залишався узгодженим.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def increment_counter(runtime: ToolRuntime) -> Command:
    """Збільшити session_count у стані агента на 1."""
    new_count = runtime.state.get("session_count", 0) + 1
    # ToolMessage з правильним tool_call_id закриває виклик інструмента
    # у списку messages; без нього наступний виклик LLM може впасти.
    return Command(update={
        "session_count": new_count,
        "messages": [ToolMessage(
            content=f"Лічильник: {new_count}",
            tool_call_id=runtime.tool_call_id,
        )],
    })

# Приклад інтеграції: create_agent(model=configured_model,
#                                  tools=[increment_counter], state_schema=CustomState)

### 15. Доступ до runtime контексту

Інструмент з доступом до runtime контексту


In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_runtime_info(runtime: ToolRuntime) -> str:
    """Показати безпечну службову інформацію про поточний виклик інструмента."""
    # Не виводимо весь runtime: він може містити конфігурацію або метадані.
    return f"Tool call ID: {runtime.tool_call_id}"

# ToolRuntime автоматично доступний інструменту, коли його викликає агент.

### 16. Базова робота зі сховищем

Інструменти можуть читати та записувати дані через `runtime.store`, якщо `store` підключено при створенні агента. `InMemoryStore` — навчальне сховище в оперативній пам'яті, не постійна база даних.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

# InMemoryStore призначений для демонстрації; дані зникають після завершення сесії.
demo_store = InMemoryStore()

@tool
def save_data(key: str, value: str, runtime: ToolRuntime) -> str:
    """Зберегти текст у підключеному до агента сховищі."""
    if runtime.store is None:
        return "Сховище не підключене"
    runtime.store.put(("data",), key, {"value": value})
    return f"Збережено: {key}"

@tool
def load_data(key: str, runtime: ToolRuntime) -> str:
    """Завантажити текст із підключеного до агента сховища."""
    if runtime.store is None:
        return "Сховище не підключене"
    item = runtime.store.get(("data",), key)
    return str(item.value.get("value", "Не знайдено")) if item else "Не знайдено"

storage_agent = create_agent(
    model=configured_model,
    tools=[save_data, load_data],
    store=demo_store,
)

### 17. Довготривала пам'ять з InMemoryStore

Приклад збереження інформації користувача. Профілі зберігаються в `InMemoryStore`; нижче є **детермінована перевірка** роботи сховища без витрат на LLM та опційний виклик агента. Не зберігай у такому демо реальні персональні або секретні дані.

In [ ]:
from typing import Any
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Прочитати навчальний профіль користувача зі сховища."""
    item = runtime.store.get(("users",), user_id) if runtime.store else None
    return str(item.value) if item else "Невідомий користувач"

@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Зберегти навчальний профіль користувача у сховищі."""
    if runtime.store is None:
        return "Сховище не підключене"
    runtime.store.put(("users",), user_id, user_info)
    return "Інформацію збережено"

# Це не постійна БД: InMemoryStore працює лише до перезапуску середовища.
user_store = InMemoryStore()
user_memory_agent = create_agent(
    model=configured_model,
    tools=[get_user_info, save_user_info],
    store=user_store,
)

Тестування збереження даних користувача


In [ ]:
# Для надійного демо спочатку зберігаємо дані без неоднозначності tool selection.
# Це тест саме сховища: агенту окремо доступні ці ж операції через tools.
user_store.put(("users",), "user123", {"name": "Іван", "age": 30})
print("Профіль user123 збережено")

# Опційний тест tool calling через LLM (платний API-запит):
# save_result = user_memory_agent.invoke({
#     "messages": [{"role": "user", "content":
#                   "Виклич save_user_info для user123: name Іван, age 30"}]
# })

Тестування отримання даних користувача


In [ ]:
# Тест читання без LLM — перевіряємо, що саме збережено в user_store.
item = user_store.get(("users",), "user123")
print(item.value if item else "Не знайдено")

# Опційний тест через LLM і tool calling:
# user_result = user_memory_agent.invoke({
#     "messages": [{"role": "user", "content": "Отримай інформацію про user123"}]
# })
# print(user_result["messages"][-1].content)

## Підключення агента до інтернету

Використовуємо пакет `langchain-tavily` та `TavilySearch`. Вебпошук реальний лише після розкоментування `internet_agent.invoke()`; на відміну від навчального `search_demo`, він потребує ключа Tavily, доступу до мережі й API-квоти.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

# Пакет langchain-tavily встановлено в першій комірці.
# API-ключі беруться автоматично з os.environ (Google Colab Secrets).
# Не передавай "ваш_openai_ключ" або "ваш_tavily_ключ" як рядки.
web_model = ChatOpenAI(model=MODEL_NAME)
web_search_tool = TavilySearch(max_results=5)

internet_agent = create_agent(
    model=web_model,
    tools=[web_search_tool],
    system_prompt=("Ти — корисний асистент. Для актуальних фактів використовуй "
                   "Tavily Search. Вказуй джерела, якщо вони доступні."),
)

# Пошук потребує доступу до інтернету та витрачає API-квоти OpenAI і Tavily.
# Розкоментуй для реального виконання:
# news_result = internet_agent.invoke({
#     "messages": [{"role": "user", "content":
#                   "Які останні новини про штучний інтелект? Додай посилання."}]
# })
# print(news_result["messages"][-1].content)

print("Internet agent створено. Для пошуку розкоментуй виклик у цій комірці.")

## Підсумок і перевірки

- **Безпечне створення агентів:** `create_agent(model=..., tools=...)`.
- **Динамічний вибір моделі:** `@wrap_model_call` + `request.override(model=...)`.
- **Інструменти:** `@tool`, валідація аргументів, `ToolRuntime` для стану та сховища.
- **Пам’ять:** `InMemorySaver` — історія потоку, `InMemoryStore` — дані за ключами; обидва лише в пам’яті поточного процесу.
- **Структурований результат:** Pydantic-схема через `response_format`.
- **Безпека:** жодного `eval()` для довільних виразів; жодних API-ключів у коді.

**Джерела для звірки API:**
- [LangChain — Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain — Middleware](https://docs.langchain.com/oss/python/langchain/middleware)
- [LangChain Tavily](https://reference.langchain.com/python/langchain-tavily)

**Обмеження перевірки:** без API-ключів власника неможливо гарантувати виконання зовнішніх викликів у його Colab; синтаксис і локальні операції перевіряються окремо.